# 1. Setup Library


In [16]:
import cv2
import numpy as np
import os
from PIL import Image
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torch.nn.parallel import DataParallel
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from torchvision.utils import save_image
import time
from tqdm import tqdm
from models import *
from utils import *
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Configurations

In [17]:
test_quality = [40]
num_imgs = 0
mode = 1
if mode == 0:
    quality_subs =   ['ESR', 'QESR', 'SRQE']
    detect_subs = ['', '_ESR', '_QESR', '_SRQE', 'full']
    loss_subs = ['HumanLoss', "MachineLoss", "TotalLoss"]
if mode == 1:
    quality_subs = ['SRQE',]
    detect_subs = ['_SRQE',]   
    loss_subs = ['HumanLoss']



In [18]:
hr_img_path = f'output/test_600/images'
hr_label_path = f'output/test_600/labels'
iqe_types = 'small'
imgsz = 64
# iqe = IQE().to(device)
if iqe_types=='small':
    iqe = Enhancer().to(device)
else:
    iqe = Enhancer(in_nc=3, out_nc=3,nf=64, level=2, num_blocks=[2, 4, 4]).to(device)
isr = ESR(scale_factor=4, use_canny=True).to(device)


In [19]:
from torchmetrics import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

def calculate_metrics(img1, img2, max_pixel_value=1.0):
    psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    psnr_value = psnr(img1, img2)
    ssim_value = ssim(img1, img2)
    return psnr_value, ssim_value



# 3. Test Generation

In [20]:
for quality in test_quality:
    for sub in quality_subs:
        for loss in loss_subs:
            print(f"Running {sub} with {loss} on quality {quality}...")
            hr_image_dir = hr_img_path
            lr_image_dir = f'output/test_{quality}_150/images'
            # lr_image_dir = hr_image_dir
            output_image_dir = f'output/E2E_Loss_Training/test_{quality}_150_{sub}_{loss}/images'
            os.makedirs(output_image_dir, exist_ok = True)
            # Duyệt qua các ảnh trong thư mục
            lr_image_files = os.listdir(lr_image_dir)
            hr_image_files = os.listdir(hr_image_dir)
            lr_image_files.sort()
            hr_image_files.sort()
        
            psnr_dict = {
                "mouse_bite" :[],
                "spur_":[], 
                "missing_hole":[],
                "short":[],
                "open_circuit":[],
                "spurious_copper":[]
        
            }
            ssim_dict = {
                "mouse_bite" :[],
                "spur_":[], 
                "missing_hole":[],
                "short":[],
                "open_circuit":[],
                "spurious_copper":[]
        
            }
            ckp_path = os.path.join('exp', f'{sub}_{loss}', 'best_weight.pth')
            if os.path.exists(ckp_path):
                ckp = torch.load(ckp_path)
                if 'isr' in ckp:
                    isr.load_state_dict(ckp['isr'])
                if 'iqe' in ckp:
                    iqe.load_state_dict(ckp['iqe'])
            start = time.time()
            transform = transforms.ToTensor()
            
            with torch.no_grad():
                for lr_image_file, hr_image_file in tqdm(zip(lr_image_files, hr_image_files), unit = 'img'):
                    # Đường dẫn đến ảnh
                    
                    lr_image_path = os.path.join(lr_image_dir, lr_image_file)
                    hr_image_path = os.path.join(hr_image_dir, hr_image_file)
                    output_image_path = os.path.join(output_image_dir, hr_image_file)
        
                    # Tải và chuyển đổi ảnh
                    lr_image = Image.open(lr_image_path).convert('RGB')
                    hr_image = Image.open(hr_image_path).convert('RGB')
                    hr_image = hr_image.resize((608, 608))
                    lr_image = lr_image.resize((152, 152))
                    if sub == 'IQE':
                        hr_image = hr_image.resize((152, 152))
                    # lr_image = Image.open(lr_image_path).convert('RGB')
        
                    lr_image = transform(lr_image).unsqueeze(0).to(device)  # Thêm batch dimension và chuyển sang GPU
                    hr_image = transform(hr_image).unsqueeze(0).to(device)  # Thêm batch dimension và chuyển sang GPU
                    # if model == bicubic:
                    # Dự đoán
                    if sub == 'ESR':
                        output = isr(lr_image)
                    elif sub == 'SRQE':
                        output = iqe(isr(lr_image))
                    elif sub == 'QESR':
                        output = isr(iqe(lr_image))
                    
                    psnr,ssim = calculate_metrics(output, hr_image)
                    for key in psnr_dict.keys():
                        if key in lr_image_path:
                            psnr_dict[key].append(psnr)
                            ssim_dict[key].append(ssim)
                            break

                    # Chuyển đổi tensor đầu ra thành ảnh và lưu
                    output_image = output.squeeze(0).to(device)  # Loại bỏ batch dimension và chuyển tensor sang CPU
                    output_image = transforms.ToPILImage()(output_image)  # Chuyển tensor thành ảnh PIL
                    output_image.save(output_image_path, )  # Lưu ảnh
            avg_psnr = [0, 0, 0, 0, 0, 0]
            avg_ssim = [0, 0, 0, 0, 0, 0]


            # Tính toán PSNR trung bình
            avg_psnr[0] = sum(psnr_dict['mouse_bite'])/len(psnr_dict['mouse_bite']) #mousebite_psnr
            avg_psnr[1] = sum(psnr_dict['spur_'])/len(psnr_dict['spur_']) #spur_psnr
            avg_psnr[2] = sum(psnr_dict['missing_hole'])/len(psnr_dict['missing_hole']) #missinghole_psnr 
            avg_psnr[3] = sum(psnr_dict['short'])/len(psnr_dict['short']) #short_psnr
            avg_psnr[4] = sum(psnr_dict['open_circuit'])/len(psnr_dict['open_circuit']) #opencircuit_psnr
            avg_psnr[5]= sum(psnr_dict['spurious_copper'])/len(psnr_dict['spurious_copper']) #spuriouscopper_psnr 
            average_psnr = sum(avg_psnr)/len(avg_psnr)

            avg_ssim[0] = sum(ssim_dict['mouse_bite'])/len(ssim_dict['mouse_bite']) #mousebite_ssim
            avg_ssim[1] = sum(ssim_dict['spur_'])/len(ssim_dict['spur_']) #spur_ssim
            avg_ssim[2] = sum(ssim_dict['missing_hole'])/len(ssim_dict['missing_hole']) #missinghole_ssim 
            avg_ssim[3] = sum(ssim_dict['short'])/len(ssim_dict['short']) #short_ssim
            avg_ssim[4] = sum(ssim_dict['open_circuit'])/len(ssim_dict['open_circuit']) #opencircuit_ssim
            avg_ssim[5]= sum(ssim_dict['spurious_copper'])/len(ssim_dict['spurious_copper']) #spuriouscopper_ssim  
            average_ssim = sum(avg_ssim)/len(avg_ssim)
            end = time.time()

            with open('runs/enhance_results.txt', 'a') as f:
                f.write(output_image_dir.split('/')[1] + '\n')
                f.write(f'missinghole_psnr: {avg_psnr[2]:.2f}' + '\n')
                f.write(f'mousebite_psnr: {avg_psnr[0]:.2f}' + '\n')
                f.write(f'opencircuit_psnr: {avg_psnr[4]:.2f}' + '\n')
                f.write(f'short_psnr: {avg_psnr[3]:.2f}' + '\n')
                f.write(f'spur_psnr: {avg_psnr[1]:.2f}' + '\n')
                f.write(f'spuriouscopper_psnr: {avg_psnr[5]:.2f}' + '\n')
                f.write(f'average_psnr: {average_psnr:.2f}' + '\n')
                f.write(f'time process: {end - start:.2f}' + '\n')

                f.write(f'missinghole_psnr: {avg_ssim[2]:.4f}' + '\n')
                f.write(f'mousebite_psnr: {avg_ssim[0]:.4f}' + '\n')
                f.write(f'opencircuit_psnr: {avg_ssim[4]:.4f}' + '\n')
                f.write(f'short_psnr: {avg_ssim[3]:.4f}' + '\n')
                f.write(f'spur_psnr: {avg_ssim[1]:.4f}' + '\n')
                f.write(f'spuriouscopper_psnr: {avg_ssim[5]:.4f}' + '\n')
                f.write(f'average_psnr: {average_ssim:.4f}' + '\n')
                f.write(f'time process: {end - start:.4f}' + '\n')
                f.write('\n')
                f.flush()
            source_dir = hr_label_path

            dest = f'output/E2E_Loss_Training/test_{quality}_150_{sub}_{loss}/labels'
            os.makedirs(dest, exist_ok=True)
        
            # Sao chép các file từ thư mục nguồn sang thư mục đích
            for filename in os.listdir(source_dir):
                source_file = os.path.join(source_dir, filename)
                file1 = os.path.join(dest, filename)
                shutil.copy(source_file, file1)


Running SRQE with HumanLoss on quality 40...


0img [00:00, ?img/s]/home/u9564043/.local/lib/python3.8/site-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `PeakSignalNoiseRatio` from `torchmetrics` was deprecated and will be removed in 2.0. Import `PeakSignalNoiseRatio` from `torchmetrics.image` instead.
  _future_warning(
/home/u9564043/.local/lib/python3.8/site-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `StructuralSimilarityIndexMeasure` from `torchmetrics` was deprecated and will be removed in 2.0. Import `StructuralSimilarityIndexMeasure` from `torchmetrics.image` instead.
  _future_warning(
681img [02:57,  3.44img/s]

# 4. Detection Test

In [ ]:
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import ops
from PIL import ImageDraw, ImageFont
import yaml
import torch
import numpy as np
from tqdm import tqdm
from PIL import Image
import os
import matplotlib.pyplot as plt
import random 
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader, Dataset
from ultralytics.utils import ops

yolov8 = YOLO('exp/weights/bestyolov8.pt').model
detection = yolov8

class_names = {
    0: 'mouse_bite',
    1: 'spur',
    2: 'missing_hole',
    3: 'short',
    4: 'open_circuit',
    5: 'spurious_copper'
}
label_colors = {
    0: (255, 0, 0),  # Red
    1: (0, 255, 0),  # Green
    2: (0, 0, 255),  # Blue
    3: (255, 255, 0),  # Yellow
    4: (255, 0, 255),  # Magenta
    5: (0, 255, 255)   # Cyan
}
def draw_and_save_predictions(image, boxes, labels, scores, class_names, save_path=None, font_path=None):
    """
    Vẽ bounding box, nhãn và độ tự tin lên ảnh, sau đó lưu ảnh nếu cần.
    
    Args:
        image (PIL.Image.Image): Ảnh đầu vào.
        boxes (torch.Tensor): Tensor chứa các bounding box, định dạng [x1, y1, x2, y2].
        labels (torch.Tensor): Tensor chứa nhãn các bounding box.
        scores (torch.Tensor): Tensor chứa điểm tự tin của các bounding box.
        class_names (dict): Mapping từ ID nhãn sang tên lớp.
        save_path (str): Đường dẫn để lưu ảnh (nếu không truyền, ảnh sẽ không được lưu).
        font_path (str): Đường dẫn tới file font TrueType (nếu không truyền sẽ dùng font mặc định).
    
    Returns:
        PIL.Image.Image: Ảnh đã được vẽ bounding box.
    """
    # Tạo bản sao ảnh để vẽ
    draw_image = image.copy()
    draw = ImageDraw.Draw(draw_image)
    
    # Tải font (nếu có)
    if font_path:
        try:
            font = ImageFont.truetype(font_path, size=20)
        except Exception as e:
            print(f"Không thể tải font từ {font_path}. Sử dụng font mặc định.")
            font = ImageFont.load_default()
    else:
        font = ImageFont.load_default()
    
    # Vẽ từng bounding box
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        label_text = f"{class_names.get(label.item(), 'Unknown')} {score:.2f}"
        color = label_colors.get(label.item())
        # Vẽ hình chữ nhật
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        
        # Vẽ nhãn với nền
        if hasattr(draw, "textbbox"):
            text_bbox = draw.textbbox((x1, y1), label_text, font=font)
            text_width, text_height = text_bbox[2] - text_bbox[0], text_bbox[3] - text_bbox[1]
        # else:
            # text_width, text_height = draw.textsize(label_text, font=font)
        draw.rectangle(
            [x1, y1 - text_height, x1 + text_width, y1],
            fill=color
        )
        draw.text((x1, y1 - text_height), label_text, fill="white", font=font)
    
    # Lưu ảnh nếu `save_path` được cung cấp
    if save_path:
        draw_image.save(save_path)


def yolo_to_xyxy(bboxes, img_w, img_h):
    """
    Convert YOLO-format [cx, cy, w, h] normalized -> [x1, y1, x2, y2] absolute
    """
    cx, cy, w, h = bboxes[:, 0], bboxes[:, 1], bboxes[:, 2], bboxes[:, 3]
    x1 = (cx - w/2) * img_w
    y1 = (cy - h/2) * img_h
    x2 = (cx + w/2) * img_w
    y2 = (cy + h/2) * img_h
    return torch.stack([x1, y1, x2, y2], dim=1)

def run_inference(image, model):
    results = model.predict(image, verbose=False)
    predictions = results[0].boxes
    # Convert to numpy for WBF compatibility
    boxes = predictions.xyxy.cpu().numpy()
    scores = predictions.conf.cpu().numpy()
    labels = predictions.cls.cpu().numpy()
    return boxes, scores, labels

def normalize_boxes(boxes, image_size):
    """Normalize box coordinates to [0, 1] range"""
    width, height = image_size
    normalized_boxes = boxes.copy()
    normalized_boxes[:, [0, 2]] /= width
    normalized_boxes[:, [1, 3]] /= height
    return normalized_boxes

def denormalize_boxes(boxes, image_size):
    """Convert normalized boxes back to pixel coordinates"""
    width, height = image_size
    denormalized_boxes = boxes.copy()
    denormalized_boxes[:, [0, 2]] *= width
    denormalized_boxes[:, [1, 3]] *= height
    return denormalized_boxes

def ensemble_inference(image, method):
    # Get predictions from both models
    boxes_yolov8, scores_yolov8, labels_yolov8 = run_inference(image, yolov8)
    boxes_yolov9, scores_yolov9, labels_yolov9 = run_inference(image, yolov9)
    # print(boxes_yolov8)
    # Get image size
    if isinstance(image, torch.Tensor):
        height, width = image.shape[-2:]
    else:  # PIL Image
        width, height = image.size
    image_size = (width, height)
    if method == 'yolov8':
        boxes, scores, labels = boxes_yolov8, scores_yolov8, labels_yolov8
    else:
        boxes, scores, labels = boxes_yolov9, scores_yolov9, labels_yolov9
   
    # Convert back to torch tensors
    boxes = torch.from_numpy(boxes).float()
    scores = torch.from_numpy(scores).float()
    labels = torch.from_numpy(labels).long()
    return boxes, scores, labels

def read_label_file(label_path, img_width, img_height):
    boxes = []
    labels = []
    with open(label_path, "r") as f:
        for line in f:
            box = list(map(float, line.strip().split()))
            labels.append(int(box[0]))
            boxes.append(yolo_to_xyxy(box, img_width, img_height))
    return boxes, labels

log_fp = open('runs/detect_results.txt', 'a')

for quality in test_quality:
    for sub in detect_subs:
        for loss in loss_subs:
            base_output_dir = "runs"
            os.makedirs(base_output_dir, exist_ok=True)
            path = f'output/E2E_Loss_Training/test_{quality}_150_{sub}_{loss}'

            if sub == 'full':
                path = f'output/test_{quality}_600' 
            data = {
                'train': f'/kaggle/input/tesstt/images',
                'val': f'{path}/images',
                'nc': 6,
                'names': {
                    0: 'mouse_bite',
                    1: 'spur',
                    2: 'missing_hole',
                    3: 'short',
                    4: 'open_circuit',
                    5: 'spurious_copper'
                }
            }
            
            with open('output/data.yaml', 'w') as file:
                yaml.dump(data, file, default_flow_style=False)
        
            print(f"Evaluating {path} dataset at QF {quality}...")
            log_fp.write(f"Evaluating {path} dataset at QF {quality}..\n")
            log_fp.flush()

            images_dir = f"{path}/images"
            images_files = sorted(os.listdir(images_dir))
            labels_dir = f"{path}/labels"
            labels_files = sorted(os.listdir(labels_dir))

            valid_dataset = YOLOTestDataset(images_dir, labels_dir)
            valid_loader = DataLoader(valid_dataset, collate_fn=yolo_collate_fn)
            map_metric = MeanAveragePrecision(iou_thresholds=[x/100 for x in range(50, 100, 5)], iou_type="bbox", class_metrics=True)        
            detection.to(device)
            detection.eval()
            map_metric.reset()
            print(f"Running inference with on {sub}...")
            # Tạo thư mục riêng cho từng phương pháp
            method_output_dir = os.path.join(base_output_dir, sub)
            os.makedirs(method_output_dir, exist_ok=True)

            pbar = tqdm(valid_loader, desc=f'Val on {sub}', unit='batch', leave=False)
            for i, (hr_images, labels) in enumerate(pbar):
                hr_images = hr_images.to(device)  # (B T C H W)s

                pred = detection(hr_images)    

                preds_for_metric = ops.non_max_suppression(pred,
                                                        conf_thres=0.25, # low conf for mAP
                                                        iou_thres=0.5,
                                                        agnostic=False,
                                                        max_det=300,
                                                        nc=data['nc'])
                predictions, targets = post_process(preds_for_metric, labels, 608, 608)             
                # print(f'pred: {predictions}\nlabels: {targets}')

                map_metric.update(predictions, targets)    

                save_path = os.path.join(method_output_dir, images_files[i])  # Lưu với cùng tên ảnh
                draw_and_save_predictions(
                    image=transforms.ToPILImage()(hr_images.squeeze(0).cpu()).copy(),
                    boxes=predictions[0]['boxes'].numpy(),
                    labels=predictions[0]['labels'].numpy(),
                    scores=predictions[0]['scores'].numpy(),
                    class_names=data['names'],
                    save_path=save_path
                )

            # Tính toán kết quả sau khi xử lý tất cả ảnh
            results = map_metric.compute()
            print(f"\nResults for on {sub}:")
            log_fp.write(f"\nResults for on {sub}:\n")
            map50_per_class = results['map_per_class']
            for class_idx, map_value in enumerate(map50_per_class):
                print(f"Class {class_idx}: mAP@50 = {map_value.item():.3f}")
                log_fp.write(f"Class {class_idx}: mAP@50 = {map_value.item():.3f}\n")
                log_fp.flush()
            print(results['map'])
            log_fp.write(f"Average: mAP50: [{results['map_50']:.3f}], mAP50-95: [{results['map']:.2f}]\n")
            log_fp.flush()
            print("-" * 50)
